In [ ]:
import pandas as pd
from google.colab import files

uploaded = files.upload()
df = pd.read_csv('sample_dirty_text.csv')

Saving sample_dirty_text.csv to sample_dirty_text.csv


In [ ]:
print(df.head())

                     noisy_text
0        This JOB is AMAZING!!!
1         Worst. Manager. EVER.
2  pay is good, but hours? meh.
3     THE benefits are... okay.
4    I just don't know anymore.


In [ ]:
print(df.shape)

(10, 1)


In [ ]:
import nltk
import string
from nltk.corpus import stopwords
nltk.download('stopwords')

def clean_text(text):
    if pd.isnull(text):
        return ""
    tokens = text.lower().translate(str.maketrans('', '', string.punctuation)).split()
    return ' '.join([word for word in tokens if word not in stopwords.words('english')])

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
df['cleaned_text'] = df['noisy_text'].apply(clean_text)
print(df.head(10))

                           noisy_text             cleaned_text
0              This JOB is AMAZING!!!              job amazing
1               Worst. Manager. EVER.       worst manager ever
2        pay is good, but hours? meh.       pay good hours meh
3           THE benefits are... okay.            benefits okay
4          I just don't know anymore.        dont know anymore
5         Loved it! Will come back :)          loved come back
6            meh. it’s fine, I guess.      meh it’s fine guess
7      great TEAM!! love the support.  great team love support
8               Too many rules! why??               many rules
9  meh, not bad but not great either.     meh bad great either


Row 6 still contained an apostrophe after cleaning. I suspected it might be an encoding issue, so I tried replacing a common corrupted character pattern (â€™)

In [ ]:
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = text.replace('â€™', "'")
    tokens = text.lower().translate(str.maketrans('', '', string.punctuation)).split()
    return ' '.join([word for word in tokens if word not in stopwords.words('english')])

In [ ]:
df['cleaned_text'] = df['noisy_text'].apply(clean_text)
print(df.head(10))

                           noisy_text             cleaned_text
0              This JOB is AMAZING!!!              job amazing
1               Worst. Manager. EVER.       worst manager ever
2        pay is good, but hours? meh.       pay good hours meh
3           THE benefits are... okay.            benefits okay
4          I just don't know anymore.        dont know anymore
5         Loved it! Will come back :)          loved come back
6            meh. it’s fine, I guess.      meh it’s fine guess
7      great TEAM!! love the support.  great team love support
8               Too many rules! why??               many rules
9  meh, not bad but not great either.     meh bad great either


The above fix didn’t resolve it, so I investigated further using repr() and character codes to see exactly what character was present

In [ ]:
print(repr(df['noisy_text'][6]))

'meh. it’s fine, I guess.'


In [ ]:
for char in df['noisy_text'][6]:
  print(char, ord(char))

m 109
e 101
h 104
. 46
  32
i 105
t 116
’ 8217
s 115
  32
f 102
i 105
n 110
e 101
, 44
  32
I 73
  32
g 103
u 117
e 101
s 115
s 115
. 46
m 109
e 101
h 104
. 46
  32
i 105
t 116
’ 8217
s 115
  32
f 102
i 105
n 110
e 101
, 44
  32
I 73
  32
g 103
u 117
e 101
s 115
s 115
. 46


This revealed the apostrophe was a curly quote (Unicode U+2019), not a standard one, which explains why the first fix didn’t catch it. Updated the function accordingly

In [14]:
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = text.replace('\u2019', "'")  # fix curly apostrophe (right single quote)
    tokens = text.lower().translate(str.maketrans('', '', string.punctuation)).split()
    return ' '.join([word for word in tokens if word not in stopwords.words('english')])

In [15]:
df['cleaned_text'] = df['noisy_text'].apply(clean_text)
print(df.head(10))

                           noisy_text             cleaned_text
0              This JOB is AMAZING!!!              job amazing
1               Worst. Manager. EVER.       worst manager ever
2        pay is good, but hours? meh.       pay good hours meh
3           THE benefits are... okay.            benefits okay
4          I just don't know anymore.        dont know anymore
5         Loved it! Will come back :)          loved come back
6            meh. it’s fine, I guess.           meh fine guess
7      great TEAM!! love the support.  great team love support
8               Too many rules! why??               many rules
9  meh, not bad but not great either.     meh bad great either


In [16]:
df.to_csv('sample_cleaned_text.csv', index=False)